In [4]:
#--1--
import pandas as pd
import numpy as np
import re

In [5]:
pd.read_excel('Energy Indicators.xls')

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,NaN,NaN,Environmental Indicators: Energy,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Energy Supply and Renewable Electricity Produc...,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,Last update: December 2015
...,...,...,...,...,...,...
277,NaN,… denotes no data available.,NaN,NaN,NaN,NaN
278,NaN,NaN,NaN,NaN,NaN,NaN
279,NaN,Data Quality:,NaN,NaN,NaN,NaN
280,NaN,The data are compiled primarily from the annua...,NaN,NaN,NaN,NaN


In [6]:
energy = pd.read_excel('Energy Indicators.xls', skiprows=17, skipfooter=38)
energy = energy.drop(energy.columns[[0, 1]], axis=1)
energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']

energy.head()

,Country,Energy Supply,Energy Supply per Capita,% Renewable
0,Afghanistan,321,10,78.669280
1,Albania,102,35,100.000000
2,Algeria,1959,51,0.551010
3,American Samoa,...,...,0.641026
4,Andorra,9,121,88.695650


In [10]:
energy = energy.replace('...', np.nan)
energy['Energy Supply'] = energy['Energy Supply'] * 1000000
energy.head()

,Country,Energy Supply,Energy Supply per Capita,% Renewable
0,Afghanistan,3.210000e+14,10.0,78.669280
1,Albania,1.020000e+14,35.0,100.000000
2,Algeria,1.959000e+15,51.0,0.551010
3,American Samoa,NaN,NaN,0.641026
4,Andorra,9.000000e+12,121.0,88.695650


In [11]:
def clean_country(name):
    name = re.sub(r'\d+', '', name) 
    name = re.sub(r'\s*\(.*\)', '', name) 
    return name.strip() 

energy['Country'] = energy['Country'].apply(clean_country)


In [22]:
country_mapping = {
    "Republic of Korea": "South Korea",
    "United States of America": "United States",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "China, Hong Kong Special Administrative Region": "Hong Kong"
}
energy['Country'] = energy['Country'].replace(country_mapping)


In [23]:
check_countries = ['South Korea', 'United States', 'United Kingdom', 'Hong Kong', 'Bolivia', 'Switzerland']
energy[energy['Country'].isin(check_countries)]

,Country,Energy Supply,Energy Supply per Capita,% Renewable
24,Bolivia,3.360000e+14,32.0,31.477120
43,Hong Kong,5.850000e+14,82.0,0.000000
164,South Korea,1.100700e+16,221.0,2.279353
197,Switzerland,1.113000e+15,136.0,57.745480
214,United Kingdom,7.920000e+15,124.0,10.600470
216,United States,9.083800e+16,286.0,11.570980


In [35]:
GDP = pd.read_excel('world_bank.xls', skiprows=3)

In [36]:
GDP = GDP.rename(columns={'Country Name': 'Country'})

In [37]:
gdp_mapping = {
    "Korea, Rep.": "South Korea",
    "Iran, Islamic Rep.": "Iran",
    "Hong Kong SAR, China": "Hong Kong"
}
GDP['Country'] = GDP['Country'].replace(gdp_mapping)

In [38]:
columns_to_keep = ['Country'] + [str(year) for year in range(2006, 2016)]
GDP = GDP[columns_to_keep]
GDP.head()

,Country,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
0,Aruba,2.469832e+09,2.677654e+09,2.843017e+09,2.553631e+09,2.453631e+09,2.637989e+09,2.615084e+09,2.727933e+09,2.791061e+09,2.963128e+09
1,Africa Eastern and Southern,5.775869e+11,6.628680e+11,7.105362e+11,7.219012e+11,8.635195e+11,9.678246e+11,9.753548e+11,9.859871e+11,1.006526e+12,9.273485e+11
2,Afghanistan,6.971383e+09,9.715765e+09,1.024977e+10,1.215484e+10,1.563384e+10,1.819041e+10,2.020357e+10,2.056449e+10,2.055058e+10,1.999814e+10
3,Africa Western and Central,3.969210e+11,4.654855e+11,5.677912e+11,5.083627e+11,5.985216e+11,6.820159e+11,7.375895e+11,8.339481e+11,8.943225e+11,7.686447e+11
4,Angola,5.238103e+10,6.526642e+10,8.853866e+10,7.030720e+10,8.169953e+10,1.094366e+11,1.249982e+11,1.334016e+11,1.372444e+11,8.721930e+10


In [40]:
ScimEn = pd.read_excel('scimagojr.xlsx')
ScimEn.head()

,Rank,Country,Region,Documents,Citable documents,Citations,Self-citations,Citations per document,H index
0,1,China,Asiatic Region,273437,272374,2336764,1615239,8.55,245
1,2,United States,Northern America,175891,172431,2230544,724472,12.68,363
2,3,India,Asiatic Region,55082,53775,463165,162944,8.41,181
3,4,Japan,Asiatic Region,50523,50065,488062,119930,9.66,193
4,5,United Kingdom,Western Europe,43389,42284,615670,111290,14.19,226


In [41]:
ScimEn_top15 = ScimEn[ScimEn['Rank'] <= 15]

In [42]:
df_merge1 = pd.merge(ScimEn_top15, energy, how='inner', on='Country')

In [43]:
final_df = pd.merge(df_merge1, GDP, how='inner', on='Country')

In [44]:
final_df = final_df.set_index('Country')

In [46]:
if 'Region' in final_df.columns:
    final_df = final_df.drop(columns=['Region'])

In [48]:
print("Розмір:", final_df.shape)
final_df

Розмір: (15, 20)


,Rank,Documents,Citable documents,Citations,Self-citations,Citations per document,H index,Energy Supply,Energy Supply per Capita,% Renewable,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,,,,,,,,,,,,,,,,,,,,
China,1,273437,272374,2336764,1615239,8.55,245,1.271910e+17,93.0,19.754910,2.752119e+12,3.550328e+12,4.594337e+12,5.101691e+12,6.087192e+12,7.551545e+12,8.532186e+12,9.570471e+12,1.047562e+13,1.106157e+13
United States,2,175891,172431,2230544,724472,12.68,363,9.083800e+16,286.0,11.570980,1.381559e+13,1.447423e+13,1.476986e+13,1.447806e+13,1.504896e+13,1.559973e+13,1.625397e+13,1.684319e+13,1.755068e+13,1.820602e+13
India,3,55082,53775,463165,162944,8.41,181,3.319500e+16,26.0,14.969080,9.402599e+11,1.216736e+12,1.198895e+12,1.341888e+12,1.675616e+12,1.823052e+12,1.827638e+12,1.856721e+12,2.039126e+12,2.103588e+12
Japan,4,50523,50065,488062,119930,9.66,193,1.898400e+16,149.0,10.232820,4.601663e+12,4.579750e+12,5.106679e+12,5.289494e+12,5.759072e+12,6.233147e+12,6.272363e+12,5.212328e+12,4.896994e+12,4.444931e+12
United Kingdom,5,43389,42284,615670,111290,14.19,226,7.920000e+15,124.0,10.600470,2.709978e+12,3.092996e+12,2.931684e+12,2.417566e+12,2.491397e+12,2.666403e+12,2.706341e+12,2.786315e+12,3.065223e+12,2.934858e+12
Germany,6,38739,38013,433148,95145,11.18,196,1.326100e+16,165.0,17.901530,2.994704e+12,3.425578e+12,3.745264e+12,3.411261e+12,3.399668e+12,3.749315e+12,3.527143e+12,3.733805e+12,3.889093e+12,3.357586e+12
Russian Federation,7,36735,36560,115938,54993,3.16,90,3.070900e+16,214.0,17.288680,9.899321e+11,1.299703e+12,1.660848e+12,1.222646e+12,1.524917e+12,2.045923e+12,2.208294e+12,2.292470e+12,2.059242e+12,1.363482e+12
Canada,8,33472,32863,568080,100953,16.97,227,1.043100e+16,296.0,61.945430,1.319265e+12,1.468820e+12,1.552990e+12,1.374625e+12,1.617343e+12,1.793327e+12,1.828366e+12,1.846597e+12,1.805750e+12,1.556509e+12
Italy,9,27983,26940,352993,87828,12.61,166,6.530000e+15,109.0,33.667230,1.949552e+12,2.213102e+12,2.408655e+12,2.199929e+12,2.136100e+12,2.294994e+12,2.086958e+12,2.141924e+12,2.162010e+12,1.836638e+12


In [62]:
import pandas as pd
import numpy as np
import re

def answer_one():

    energy = pd.read_excel('Energy Indicators.xls', skiprows=17, skipfooter=38)
    energy = energy.drop(energy.columns[[0, 1]], axis=1)
    energy.columns = ['Country', 'Energy Supply', 'Energy Supply per Capita', '% Renewable']
    energy = energy.replace('...', np.nan)
    energy['Energy Supply'] = energy['Energy Supply'] * 1000000
    
    def clean_country(name):
        name = re.sub(r'\d+', '', name)
        name = re.sub(r'\s*\(.*\)', '', name)
        return name.strip()
        
    energy['Country'] = energy['Country'].apply(clean_country)
    
    country_mapping = {
        "Republic of Korea": "South Korea",
        "United States of America": "United States",
        "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
        "China, Hong Kong Special Administrative Region": "Hong Kong"
    }
    energy['Country'] = energy['Country'].replace(country_mapping)

    GDP = pd.read_excel('world_bank.xls', skiprows=3) 
    GDP = GDP.rename(columns={'Country Name': 'Country'})
    gdp_mapping = {
        "Korea, Rep.": "South Korea",
        "Iran, Islamic Rep.": "Iran",
        "Hong Kong SAR, China": "Hong Kong"
    }
    GDP['Country'] = GDP['Country'].replace(gdp_mapping)
    columns_to_keep = ['Country'] + [str(year) for year in range(2006, 2016)]
    GDP = GDP[columns_to_keep]

    ScimEn = pd.read_excel('scimagojr.xlsx') 

    ScimEn_top15 = ScimEn[ScimEn['Rank'] <= 15]
    df_merge1 = pd.merge(ScimEn_top15, energy, how='inner', on='Country')
    final_df = pd.merge(df_merge1, GDP, how='inner', on='Country')
    
    final_df = final_df.set_index('Country')
    return final_df

answer_one()

C:\Users\PC\AppData\Local\Temp\ipykernel_16620\606840000.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy = energy.replace('...', np.nan)


,Rank,Region,Documents,Citable documents,Citations,Self-citations,Citations per document,H index,Energy Supply,Energy Supply per Capita,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,,,,,,,,,,,,,,,,,,,,,
China,1,Asiatic Region,273437,272374,2336764,1615239,8.55,245,1.271910e+11,93.0,...,2.752119e+12,3.550328e+12,4.594337e+12,5.101691e+12,6.087192e+12,7.551545e+12,8.532186e+12,9.570471e+12,1.047562e+13,1.106157e+13
United States,2,Northern America,175891,172431,2230544,724472,12.68,363,9.083800e+10,286.0,...,1.381559e+13,1.447423e+13,1.476986e+13,1.447806e+13,1.504896e+13,1.559973e+13,1.625397e+13,1.684319e+13,1.755068e+13,1.820602e+13
India,3,Asiatic Region,55082,53775,463165,162944,8.41,181,3.319500e+10,26.0,...,9.402599e+11,1.216736e+12,1.198895e+12,1.341888e+12,1.675616e+12,1.823052e+12,1.827638e+12,1.856721e+12,2.039126e+12,2.103588e+12
Japan,4,Asiatic Region,50523,50065,488062,119930,9.66,193,1.898400e+10,149.0,...,4.601663e+12,4.579750e+12,5.106679e+12,5.289494e+12,5.759072e+12,6.233147e+12,6.272363e+12,5.212328e+12,4.896994e+12,4.444931e+12
United Kingdom,5,Western Europe,43389,42284,615670,111290,14.19,226,7.920000e+09,124.0,...,2.709978e+12,3.092996e+12,2.931684e+12,2.417566e+12,2.491397e+12,2.666403e+12,2.706341e+12,2.786315e+12,3.065223e+12,2.934858e+12
Germany,6,Western Europe,38739,38013,433148,95145,11.18,196,1.326100e+10,165.0,...,2.994704e+12,3.425578e+12,3.745264e+12,3.411261e+12,3.399668e+12,3.749315e+12,3.527143e+12,3.733805e+12,3.889093e+12,3.357586e+12
Russian Federation,7,Eastern Europe,36735,36560,115938,54993,3.16,90,3.070900e+10,214.0,...,9.899321e+11,1.299703e+12,1.660848e+12,1.222646e+12,1.524917e+12,2.045923e+12,2.208294e+12,2.292470e+12,2.059242e+12,1.363482e+12
Canada,8,Northern America,33472,32863,568080,100953,16.97,227,1.043100e+10,296.0,...,1.319265e+12,1.468820e+12,1.552990e+12,1.374625e+12,1.617343e+12,1.793327e+12,1.828366e+12,1.846597e+12,1.805750e+12,1.556509e+12
Italy,9,Western Europe,27983,26940,352993,87828,12.61,166,6.530000e+09,109.0,...,1.949552e+12,2.213102e+12,2.408655e+12,2.199929e+12,2.136100e+12,2.294994e+12,2.086958e+12,2.141924e+12,2.162010e+12,1.836638e+12


In [60]:
#--2--
def answer_two():
    Top15 = answer_one() 
    years = [str(year) for year in range(2006, 2016)]
    avgGDP = Top15[years].mean(axis=1).sort_values(ascending=False)
    return avgGDP
answer_two()

C:\Users\PC\AppData\Local\Temp\ipykernel_16620\3525874804.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy = energy.replace('...', np.nan)


Country
United States         1.570403e+13
China                 6.927707e+12
Japan                 5.239642e+12
Germany               3.523342e+12
United Kingdom        2.780276e+12
France                2.691337e+12
Italy                 2.142986e+12
Brazil                1.988889e+12
Russian Federation    1.666746e+12
Canada                1.616359e+12
India                 1.602352e+12
Spain                 1.400886e+12
South Korea           1.221372e+12
Australia             1.207513e+12
Iran                  4.563261e+11
dtype: float64

In [63]:
#--3--
def answer_three():
    # Отримуємо основну таблицю з усіма даними
    Top15 = answer_one()
    
    # Отримуємо наш відсортований рейтинг ВВП з попереднього кроку
    avgGDP = answer_two()
    
    # Знаходимо назву країни на 6-му місці (індекс 5, бо рахунок з нуля)
    country_6th = avgGDP.index[5]
    
    # Віднімаємо від ВВП 2015 року ВВП 2006 року для цієї країни
    gdp_change = Top15.loc[country_6th, '2015'] - Top15.loc[country_6th, '2006']
    
    return gdp_change

# Викликаємо функцію, щоб побачити результат
answer_three()

C:\Users\PC\AppData\Local\Temp\ipykernel_16620\606840000.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy = energy.replace('...', np.nan)
C:\Users\PC\AppData\Local\Temp\ipykernel_16620\606840000.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy = energy.replace('...', np.nan)


np.float64(118652421857.7959)

In [65]:
#--4--
def answer_four():
    Top15 = answer_one()
    Top15['Citation Ratio'] = Top15['Self-citations'] / Top15['Citations']
    max_ratio = Top15['Citation Ratio'].max()
    country_with_max = Top15['Citation Ratio'].idxmax()
    return (country_with_max, max_ratio)
answer_four()

C:\Users\PC\AppData\Local\Temp\ipykernel_16620\606840000.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy = energy.replace('...', np.nan)


('China', np.float64(0.6912289816173135))

In [66]:
#--5--
def answer_five():
    Top15 = answer_one()
    Top15['PopEst'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    sorted_Top15 = Top15.sort_values(by='PopEst', ascending=False)
    third_country = sorted_Top15.index[2]
    
    return third_country

answer_five()

C:\Users\PC\AppData\Local\Temp\ipykernel_16620\606840000.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy = energy.replace('...', np.nan)


'United States'

In [71]:
#--6--
def answer_six():
    Top15 = answer_one()
    Top15['PopEst'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    Top15['Citable docs per Capita'] = Top15['Citable documents'] / Top15['PopEst']
    Top15['Citable docs per Capita'] = pd.to_numeric(Top15['Citable docs per Capita'])
    Top15['Energy Supply per Capita'] = pd.to_numeric(Top15['Energy Supply per Capita'])
    
    correlation = Top15['Citable docs per Capita'].corr(Top15['Energy Supply per Capita'])
    return correlation

answer_six()

C:\Users\PC\AppData\Local\Temp\ipykernel_16620\606840000.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy = energy.replace('...', np.nan)


np.float64(0.7434709127726777)

In [77]:
answer_one()

C:\Users\PC\AppData\Local\Temp\ipykernel_16620\606840000.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy = energy.replace('...', np.nan)


,Rank,Region,Documents,Citable documents,Citations,Self-citations,Citations per document,H index,Energy Supply,Energy Supply per Capita,...,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
Country,,,,,,,,,,,,,,,,,,,,,
China,1,Asiatic Region,273437,272374,2336764,1615239,8.55,245,1.271910e+11,93.0,...,2.752119e+12,3.550328e+12,4.594337e+12,5.101691e+12,6.087192e+12,7.551545e+12,8.532186e+12,9.570471e+12,1.047562e+13,1.106157e+13
United States,2,Northern America,175891,172431,2230544,724472,12.68,363,9.083800e+10,286.0,...,1.381559e+13,1.447423e+13,1.476986e+13,1.447806e+13,1.504896e+13,1.559973e+13,1.625397e+13,1.684319e+13,1.755068e+13,1.820602e+13
India,3,Asiatic Region,55082,53775,463165,162944,8.41,181,3.319500e+10,26.0,...,9.402599e+11,1.216736e+12,1.198895e+12,1.341888e+12,1.675616e+12,1.823052e+12,1.827638e+12,1.856721e+12,2.039126e+12,2.103588e+12
Japan,4,Asiatic Region,50523,50065,488062,119930,9.66,193,1.898400e+10,149.0,...,4.601663e+12,4.579750e+12,5.106679e+12,5.289494e+12,5.759072e+12,6.233147e+12,6.272363e+12,5.212328e+12,4.896994e+12,4.444931e+12
United Kingdom,5,Western Europe,43389,42284,615670,111290,14.19,226,7.920000e+09,124.0,...,2.709978e+12,3.092996e+12,2.931684e+12,2.417566e+12,2.491397e+12,2.666403e+12,2.706341e+12,2.786315e+12,3.065223e+12,2.934858e+12
Germany,6,Western Europe,38739,38013,433148,95145,11.18,196,1.326100e+10,165.0,...,2.994704e+12,3.425578e+12,3.745264e+12,3.411261e+12,3.399668e+12,3.749315e+12,3.527143e+12,3.733805e+12,3.889093e+12,3.357586e+12
Russian Federation,7,Eastern Europe,36735,36560,115938,54993,3.16,90,3.070900e+10,214.0,...,9.899321e+11,1.299703e+12,1.660848e+12,1.222646e+12,1.524917e+12,2.045923e+12,2.208294e+12,2.292470e+12,2.059242e+12,1.363482e+12
Canada,8,Northern America,33472,32863,568080,100953,16.97,227,1.043100e+10,296.0,...,1.319265e+12,1.468820e+12,1.552990e+12,1.374625e+12,1.617343e+12,1.793327e+12,1.828366e+12,1.846597e+12,1.805750e+12,1.556509e+12
Italy,9,Western Europe,27983,26940,352993,87828,12.61,166,6.530000e+09,109.0,...,1.949552e+12,2.213102e+12,2.408655e+12,2.199929e+12,2.136100e+12,2.294994e+12,2.086958e+12,2.141924e+12,2.162010e+12,1.836638e+12


In [76]:
#--7-- 
def answer_seven():
    
    Top15 = answer_one()
    median_renew = Top15['% Renewable'].median()
    Top15['HighRenew'] = (Top15['% Renewable'] >= median_renew).astype(int)
    Top15 = Top15.sort_values(by='Rank')
    
    return Top15['HighRenew']

answer_seven()

C:\Users\PC\AppData\Local\Temp\ipykernel_16620\606840000.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy = energy.replace('...', np.nan)


Country
China                 1
United States         0
India                 0
Japan                 0
United Kingdom        0
Germany               1
Russian Federation    1
Canada                1
Italy                 1
South Korea           0
France                1
Iran                  0
Spain                 1
Brazil                1
Australia             0
Name: HighRenew, dtype: int64

In [78]:
#--8--
def answer_eight():
    Top15 = answer_one()
    ContinentDict  = {'China':'Asia', 
                      'United States':'North America', 
                      'Japan':'Asia', 
                      'United Kingdom':'Europe', 
                      'Russian Federation':'Europe', 
                      'Canada':'North America', 
                      'Germany':'Europe', 
                      'India':'Asia',
                      'France':'Europe', 
                      'South Korea':'Asia', 
                      'Italy':'Europe', 
                      'Spain':'Europe', 
                      'Iran':'Asia',
                      'Australia':'Australia', 
                      'Brazil':'South America'}
    
    Top15['PopEst'] = Top15['Energy Supply'] / Top15['Energy Supply per Capita']
    Top15['Continent'] = Top15.index.map(lambda x: ContinentDict[x])
    ans = Top15.groupby('Continent')['PopEst'].agg(['size', 'sum', 'mean', 'std'])
    
    return ans

answer_eight()

C:\Users\PC\AppData\Local\Temp\ipykernel_16620\606840000.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  energy = energy.replace('...', np.nan)


,size,sum,mean,std
Continent,,,,
Asia,5,2.898666e+09,5.797333e+08,6.790979e+08
Australia,1,2.331602e+07,2.331602e+07,NaN
Europe,6,4.579297e+08,7.632161e+07,3.464767e+07
North America,2,3.528552e+08,1.764276e+08,1.996696e+08
South America,1,2.059153e+08,2.059153e+08,NaN
